<div style="font-family: 'Times New Roman', serif;">

**[RECALL]**  
  
**Value of the underlying asset value (Assuming (Driftless) Geometric Brownian Motion)**

$$
S_t
 = S_0\exp\left\{-\frac{\sigma^2 t}{2} + \sigma W_t\right\}
  \stackrel{d}{=} S_0\exp\left\{-\frac{\sigma^2 t}{2} + \sigma Z\sqrt{t}\right\}
$$

$(W_t)_{t\geq0}$: brownian motion which satisfies:  
$Z$: standard Gaussian random variable

**Value of a European Call Option at time zero (Black-Scholes)**  

$$
\mathrm{C}^{\mathrm{BS}}(S_0, K, T;\sigma) = \mathbb{E}\Big[\max(S_{T} - K, 0)\Big]
 = S_0\Big(\mathcal{N}(d_{+}) - \mathrm{e}^{k}\mathcal{N}(d_{-})\Big),
$$
where
$$
d_{\pm} = \frac{-k}{\sigma\sqrt{T}} \pm\frac{\sigma\sqrt{T}}{2}
$$  

$T$: maturity  
$K$: strike price  
$(S_t)_{t\geq0}$: underlying asset  
$k:= log(K/S_0)$  
$\sigma$: volatility of returns of the underlying  
$\mathcal{N}(\cdot)$: CDF of the standard normal distribution  
$$\mathcal{N}(x) = \frac{1}{\sqrt{2\pi}}\int_{-\infty}^{x}\exp\left\{-\frac{z^2}{2}\right\}\mathrm{d}z$$

</div>


In [18]:
from scipy.stats import norm
import numpy as np

def blackscholes_call(S0, K, sigma, T):
    '''
    #Inputs:
    S0: initial stock price
    K: strike
    sigma: instantaneous volatility
    T: time to maturity
    #Output:
    Black-Scholes Call price
    '''

    k = np.log(K/S0)
    dp = -k/(sigma*np.sqrt(T)) + sigma*np.sqrt(T)/2
    dm = -k/(sigma*np.sqrt(T)) - sigma*np.sqrt(T)/2

    return S0*(norm.cdf(dp) - np.exp(k)*norm.cdf(dm))

<div style="font-family: 'Times New Roman', serif;">

### **Implied Volatility**

$$
\mathrm{C}^{\mathrm{BS}}(S_0, K, T;\sigma)
$$  
Among the pamameters of the payoption, following are directly observable.  
1) S0 : current price of the underlying asset
2) K (strike) / T (maturity) : fixed term in the contract  

However, $\sigma$ (volatility) is *not* directly observable.  

**Implied Volatility** ($\sigma_{\text{imp}}$) is the unique volatility value that can be obtained by the Black-Scholes equation and the current market price of the option ($C^{\text{market}}$)  
 $$C^{\text{BS}}(S_0, K, T; \sigma_{\text{imp}}) = C^{\text{market}}$$

</div>

<div style="font-family: 'Times New Roman', serif;">

#### **Bijectivity of $\sigma \mapsto C^{\text{BS}}(S_0, K, T; \sigma)$**  

**1) Uniqueness of $\sigma_{\text{imp}}$ (injectivity)**   
*two distinct volatilities does not produce the same call option price*  

**Vega**: option's sensitivity to volatility  
$$\mathcal{V} = \frac{\partial C^{\text{BS}}}{\partial \sigma} = S_0 \sqrt{T} \varphi(d_+) > 0 \quad \text{for all } \sigma \in (0, \infty)$$
* $\varphi$: probability density function of standard normal distribution.  

Since Vega is strictly positive everywhere, $C^{\text{BS}}$ is strictly increasing w.r.t. $\sigma$.  
Therefore, strictly injective.  
</div>

<div style="font-family: 'Times New Roman', serif;">

**2) Existence of $\sigma_{\text{imp}}$ (Surjectivity)**  
*The image of $(0, \infty)$ under $C^{\text{BS}}$ is equal to the codomain of $C^{\text{BS}}$*

$$\lim_{\sigma \to 0^+} C^{\text{BS}}(\sigma) = \max(S_0 - K, 0)$$

$$\lim_{\sigma \to \infty} C^{\text{BS}}(\sigma) = S_0$$

$\because \mathcal{N}(d_+) \to 1 \quad \mathcal{N}(d_-) \to 0$  

Since $C^{\text{BS}}$ is continuous on  $(0, \infty)$, *Intermediate Value Thm* ensures that the image of $(0, \infty)$ is the open interval
$$(max(S_0-K,0), S_0)$$
  
</br></br>  

Therefore, for any $C^{\text{market}}$, there exists a *unique* implied volatility  
$$\sigma_{\text{imp}} = (C^{\text{BS}})^{-1}(C^{\text{market}})$$

*However there is NO closed-form inverse of the implied volatility.*  
Therefore, we use Root finding algorithm to find the root of 
$$C^{\mathrm{BS}}(\sigma) - C^{\mathrm{market}} = 0$$
</div>

<div style="font-family: 'Times New Roman', serif;">  

#### **Root finding by Newton-Rapshon**

</div>

<div style="font-family: 'Times New Roman', serif;">  

**Newton-Rapshon**  
Given an equation: $f(x) = 0$  
Start with an initial guess $x_0$ and iteratively generate a sequence of approximations $(x_n)_{n \ge 0}$ where $$\lim_{n\to \infty} {x_n} \to x^*$$  

  
By the first-order Taylor Series expansion around $x_n$  
$$f(x) \approx f(x_n) + f'(x_n)(x - x_n)$$
  
Since we want the $x_{n+1}$ to be the value where $f(x_{n+1}) = 0$, we solve the following equation:  
$$0 = f(x_n) + f'(x_n)(x_{x+1} - x_n)$$

Therefore,  
$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

</div>

<div style="font-family: 'Times New Roman', serif;">  

*How fast would it converge? (quadratic convergence)*  
consider the second order taylor series expansion
$$0 = f(x^*) = f(x_n) + f'(x_n)(x^* - x_n) + \frac{f''(\xi_n)}{2}(x^* - x_n)^2 \quad (\xi_n \in (x_n, x^*))$$

plugging in $0 = f(x_n) + f'(x_n)(x_{x+1} - x_n)$ we get:
$$\vert{}x_{n+1} - x^*\vert{} = \left\vert\frac{f''(\xi_n)}{2f'(x_n)}\right\vert \vert{}x_n - x^*\vert^2 \leq M\vert x_n - x^*\vert^2$$
$\to$ that is, when $\vert{}x_n - x^*\vert < \frac{1}{M}$, the error shrinks quadratically  
(this is under the condition that $M <\infty$ and $f$ is twice differentiable.) 

</div>

<div style="font-family: 'Times New Roman', serif;">  

#### **Finite Differences**

**Q: How to calculate $f'(x_n)$ in the Newton-Raphson Method****
$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

definition of derivative:
$$\frac{f(x+\Delta x) - f(x)}{\Delta x} \to f'(x) \quad \text{as} \quad \Delta x \to \infty$$
 
*Instead of sending $\Delta$ to $0$, we choose a "small enough" $\Delta$.*

Consider a local taylor expansion around a point $x\in \mathbb{R}$:
$$f(x+\varepsilon) = f(x) + \varepsilon f'(x) + \mathcal{O}(\varepsilon^2)$$

$$ \iff
f'(x) = \frac{f(x+\varepsilon) - f(x)}{\varepsilon}  + \mathcal{O}(\varepsilon),
$$

$\to$ carries an error of $\mathcal{O}(\varepsilon)$   

*How to reduce the error term more significantly?*  

Evaluate on both sides of $x$:  

$$f(x + \varepsilon) = f(x) + \varepsilon f'(x) + \frac{\varepsilon^2}{2}f''(x) + \frac{\varepsilon^3}{6}f'''(\xi_1)$$
$$f(x - \varepsilon) = f(x) - \varepsilon f'(x) + \frac{\varepsilon^2}{2}f''(x) - \frac{\varepsilon^3}{6}f'''(\xi_2)$$
$$\iff f(x + \varepsilon) - f(x - \varepsilon) = 2\varepsilon f'(x) + \mathcal{O}(\varepsilon^3)$$
$$\iff f'(x) = \frac{f(x + \varepsilon) - f(x - \varepsilon)}{2\varepsilon} + \mathcal{O}(\varepsilon^2)$$
  
$\to$ carries an error of $\mathcal{O}(\varepsilon ^2)$    
</div>

In [22]:
def derivative_f(f, x, epsilon):
    """
    returns the derivative of the function f() at the point x using the central difference scheme with tolerance epsilon
    f: function to differentiate
    x: value at which to compute the derivative
    epsilon: tolerance
    
    output: f'(x)
    """

    return (f(x+epsilon)-f(x-epsilon))/(2*epsilon)


<div style="font-family: 'Times New Roman', serif;">  

#### **Stopping Criterion of newton-raphson**
**Newton-Raphson**  
$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

Suppose a tolerence level of $\varepsilon$  

Stopping criterion:  
$|f(x_n)|\leq \varepsilon$ : would indicate that $x_n$ is sufficiently close to the root of $f(x) = 0$  

*Normally we would also have to consider the step size $|x_{n+1} - x_n|$ in the case of diverging or taking forever to converge(stagnation). 
However let's assume a "well-behaved" pricing function and a "reasonable" initial guess.*

</div>

In [23]:

def newton_raphson(f, x0, epsilon, epsilon_deriv=1E-8, max_iter = 10000):
    """
    Implements Newton Raphson
    f: function to solve for
    fprime: derivative of f
    x0: starting point
    epsilon: tolerance
    epsilon_deriv: tolerance for the derivative
    
    output: x, f(x), n, where x is the solution, f(x) its value, n the total number of iterations
    """

    x = x0

    nstep = 0
    while abs(f(x)) > epsilon and nstep < max_iter:
        x = x - f(x)/derivative_f(f, x, epsilon_deriv)
        nstep += 1

    return x, f(x), nstep
    

<div style="font-family: 'Times New Roman', serif;">  

#### **Returning to implied volatility**

we wish to find a root of:
$$C^{\mathrm{BS}}(\sigma) - C^{\mathrm{market}} = 0$$
</div>

In [ ]:
S0, K, T = 100., 100., 1.
optionPrice = 3.9

f = lambda sig: blackscholes_call(S0, K, sig, T) - optionPrice

impliedVol, f_val, nstep = newton_raphson(f, x0 = 0.3, epsilon = 1E-10)
print("Implied volatility: %.2f" %(100.*impliedVol), "%")


Implied volatility: 9.78 %
